# HAM10000 — Framework Interpretável para Classificação de Lesões de Pele

Implementação de referência do artigo **"A Multi-Level Interpretable Framework
for Skin Cancer Classification Using Deep Learning and Fuzzy Systems"**.

Pipeline: EfficientNet-B0 (fine-tuned) → Grad-CAM → extração das 4 features
ABCD (Asymmetry, Border, Color, differential Structures) → Sistema de
Inferência Fuzzy Mamdani de 5 entradas (A, B, C, D, P) → score de risco
interpretável em `[0, 10]` com explicação em linguagem natural.

## Changelog v2 → v3

| # | Local | Problema v2 | Correção v3 |
|---|---|---|---|
| 1 | Calibração/inferência | `A = p_mal × 0.3` — proxy redundante com P | `A = sA(mask)` — assimetria geométrica real |
| 2 | Funções de pertinência (entrada) | `tl/th` criavam zonas mortas entre termos | `overlap = span×0.25` garante sobreposição explícita |
| 3 | Função de pertinência (saída) | Lacunas e sobreposições inconsistentes em `[0,10]` | Sobreposição uniforme de 1.5pt em todos os termos |
| 4 | Fallback de inferência | Divisor "magic number" (8.9) | Pesos somam 10, resultado direto em `[0,10]` |
| 5 | Cross-validation | Mesmo proxy `p_mal×0.3` e `build_fuzzy_v2` | A real + `build_fuzzy_v3` em cada fold |
| 6 | Avaliação no teste | Funções `inferir/regras/interp` v2; coluna `A_raw` | v3; coluna `A_real` |
| 7 | Dashboard | Labels "Fuzzy v2", "A proxy" | "Fuzzy v3", "A real" |
| 8 | Relatório XLSX | "A raw/proxy", "Regras Fuzzy v2" | "A real (sA CAM)", "Regras Fuzzy v3" |
| 9 | Figura Grad-CAM | `suptitle(...,...)` — `TypeError` por `Ellipsis` | `suptitle(..., fontsize=13)` |
| 10 | Figura Grad-CAM | Linha de score solta sem inferência | `inferir_v3()` chamado corretamente |
| 11 | — | Células de teste e vazias remanescentes | Removidas |

> ⚠️ **Nota de reprodutibilidade:** a correção #1 (assimetria real em vez do
> proxy) é uma melhoria metodológica genuína, mas desacopla `A` de `P` —
> como a rule base de 16 regras foi originalmente calibrada com o proxy
> antigo, essa mudança tende a reduzir a sensibilidade do sistema fuzzy em
> relação aos números publicados no artigo. Ver discussão na Seção 6 e nos
> resultados de cross-validation (Seção 9) deste notebook.


## 1. Configuração de paths e parâmetros do experimento

Define os caminhos do dataset HAM10000 e do checkpoint do EfficientNet-B0 já
treinado, além de dois parâmetros que controlam todo o experimento:

- `MAX_IMAGES` — quantidade de linhas do metadata CSV carregadas via `.head()`.
  **Atenção:** isso lê as *primeiras* N linhas do arquivo, não uma amostra
  aleatória do HAM10000 completo. O balanceamento maligno/benigno resultante
  depende da ordenação do CSV, não da prevalência real do dataset
  (~19,5% maligno no HAM10000 completo). Com `MAX_IMAGES=3000`, o subconjunto
  obtido fica em torno de 54% maligno / 46% benigno — note que isso é diferente
  da proporção citada no artigo (40,8%/59,2%), o que sugere que o número do
  artigo foi escrito a partir de outra execução ou não foi reconferido contra
  esta versão do código.
- `TEST_SIZE` — proporção do split de teste (20%).

**Pré-requisito:** o checkpoint `effnet_ham_best.pt` precisa já existir — este
notebook não re-treina a rede, apenas a reutiliza.


In [ ]:
METADATA_CSV = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv'
IMG_DIRS = [
    '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1',
    '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2',
]
MODEL_PT = '/kaggle/input/models/matheusbuttow/modelo-bracis/pytorch/default/1/effnet_ham_best.pt'
# Se fez upload direto: MODEL_PT = '/kaggle/working/effnet_ham_best.pt'

MAX_IMAGES = 3000
TEST_SIZE  = 0.20

import os
print(f'Modelo existe? {os.path.exists(MODEL_PT)}')
if os.path.exists(MODEL_PT):
    print(f'Tamanho: {os.path.getsize(MODEL_PT)/1024**2:.1f} MB')

In [ ]:
!pip install scikit-fuzzy openpyxl scikit-image --quiet
print('OK')

## 2. Instalação de dependências

`scikit-fuzzy` (sistema de inferência fuzzy Mamdani), `openpyxl` (geração do
relatório XLSX) e `scikit-image` (LBP e medição de regiões para o critério D).


In [ ]:
import os, cv2, numpy as np, pandas as pd, warnings
from pathlib import Path
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import transforms, models
from PIL import Image

from skimage import measure
from skimage.feature import local_binary_pattern
from sklearn.cluster import KMeans
from sklearn.preprocessing import QuantileTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

DEVICE   = 'cpu'
IMG_SIZE = 224
MALIGNANT = {'mel','bcc','akiec'}
print(f'Device: {DEVICE}')

## 3. Imports e constantes globais

- `DEVICE='cpu'` — inferência roda em CPU (Grad-CAM + segmentação não exigem GPU
  neste volume de dados; ajuste para `'cuda'` se disponível e quiser acelerar).
- `MALIGNANT = {'mel','bcc','akiec'}` — define a binarização das 7 classes
  originais do HAM10000 em maligno/benigno (ver Seção 3.1 do artigo).


In [ ]:
df_meta = pd.read_csv(METADATA_CSV).head(MAX_IMAGES).copy()
df_meta['is_malignant'] = df_meta['dx'].isin(MALIGNANT).astype(int)
img_map = {}
for pasta in IMG_DIRS:
    if os.path.exists(pasta):
        for arq in Path(pasta).glob('*.jpg'):
            img_map[arq.stem] = str(arq)
df_meta['image_path'] = df_meta['image_id'].map(img_map)
df_meta = df_meta.dropna(subset=['image_path']).reset_index(drop=True)
df_train, df_test = train_test_split(
    df_meta, test_size=TEST_SIZE,
    stratify=df_meta['is_malignant'], random_state=42)
df_train = df_train.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)
print(f'Treino: {len(df_train)} | Teste: {len(df_test)}')
print(f'Malignas treino: {df_train.is_malignant.sum()} | teste: {df_test.is_malignant.sum()}')

## 4. Carregamento do dataset e split treino/teste

1. Lê o metadata CSV (`.head(MAX_IMAGES)`) e mapeia cada `image_id` ao caminho
   físico do arquivo `.jpg` nas duas pastas de imagens do HAM10000.
2. Binariza o rótulo (`is_malignant`).
3. Split estratificado por `is_malignant`, 80/20, `random_state=42`.

Não há um terceiro split de validação separado — a seleção de hiperparâmetros
do fuzzy é validada via cross-validation sobre o próprio conjunto de treino
(Seção 9 deste notebook), não sobre um held-out adicional.


In [ ]:
def build_model(device):
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    for p in m.parameters(): p.requires_grad = False
    for p in list(m.features.parameters())[-20:]: p.requires_grad = True
    m.classifier = nn.Sequential(
        nn.Dropout(0.4), nn.Linear(1280,256), nn.ReLU(),
        nn.Dropout(0.3), nn.Linear(256,1), nn.Sigmoid())
    return m.to(device)

tfm_val = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

model = build_model(DEVICE)
model.load_state_dict(torch.load(MODEL_PT, map_location=DEVICE))
model.eval()

# Teste rápido
img_t = tfm_val(Image.open(df_test.iloc[0]['image_path']).convert('RGB')).unsqueeze(0)
with torch.no_grad():
    p_t = float(model(img_t).squeeze())
print(f'Modelo OK | P(maligno) amostra: {p_t:.4f} | real: {df_test.iloc[0]["is_malignant"]}')

## 5. Modelo: EfficientNet-B0 fine-tuned

Reconstrói a arquitetura (backbone pré-treinado em ImageNet + head binário com
duplo dropout) e carrega os pesos do checkpoint salvo. As últimas 20 camadas
convolucionais do backbone foram treináveis durante o fine-tuning original;
aqui o modelo é usado **apenas em modo de inferência** (`model.eval()`).

Um teste rápido com a primeira imagem do conjunto de teste confirma que os
pesos carregaram corretamente antes de seguir para a extração de features.


In [ ]:
class GradCAMExtractor:
    def __init__(self,model,device):
        self.model=model; self.device=device; self.grads=None; self.acts=None
        model.features[-1].register_forward_hook(
            lambda m,i,o: setattr(self,'acts',o.detach()))
        model.features[-1].register_backward_hook(
            lambda m,gi,go: setattr(self,'grads',go[0].detach()))
        self.tfm=tfm_val

    def get_cam(self,img_t):
        self.model.eval(); img_t.requires_grad_(True)
        out=self.model(img_t); self.model.zero_grad()
        out.backward(torch.ones_like(out))
        w=self.grads.mean(dim=[2,3],keepdim=True)
        cam=F.relu((w*self.acts).sum(dim=1,keepdim=True))
        cam=F.interpolate(cam,(IMG_SIZE,IMG_SIZE),mode='bilinear',align_corners=False)
        cam=cam.squeeze().detach().cpu().numpy()
        mn,mx=cam.min(),cam.max()
        return (cam-mn)/(mx-mn+1e-8)

    def segment(self,img_rgb):
        h,w=img_rgb.shape[:2]
        bgr=cv2.cvtColor(img_rgb,cv2.COLOR_RGB2BGR)
        _,otsu=cv2.threshold(cv2.GaussianBlur(
            cv2.cvtColor(img_rgb,cv2.COLOR_RGB2GRAY),(7,7),0),
            0,255,cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)
        cnts,_=cv2.findContours(otsu,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        if not cnts: return otsu.astype(bool)
        x,y,bw,bh=cv2.boundingRect(max(cnts,key=cv2.contourArea))
        mg=10; x=max(0,x-mg); y=max(0,y-mg)
        bw=min(w-x,bw+2*mg); bh=min(h-y,bh+2*mg)
        if bw<10 or bh<10: return otsu.astype(bool)
        mask_gc=np.zeros((h,w),np.uint8)
        bgm=np.zeros((1,65),np.float64); fgm=np.zeros((1,65),np.float64)
        try:
            cv2.grabCut(bgr,mask_gc,(x,y,bw,bh),bgm,fgm,5,cv2.GC_INIT_WITH_RECT)
            mf=np.where((mask_gc==2)|(mask_gc==0),0,1).astype(np.uint8)
        except: mf=otsu//255
        k=np.ones((9,9),np.uint8)
        mf=cv2.morphologyEx(mf*255,cv2.MORPH_CLOSE,k)
        mf=cv2.morphologyEx(mf,cv2.MORPH_OPEN,k)
        lb=measure.label(mf)
        if lb.max()==0: return mf.astype(bool)
        return (lb==(np.bincount(lb.flat)[1:].argmax()+1))

    def sA(self, mask):
        """
        Assimetria geometrica da lesao segmentada (criterio A do ABCD clinico).
        Compara metades espelhadas da mascara binaria nos dois eixos.
        Substitui sA(cam) anterior (r=0.04) por sA_mask (r esperado 0.35-0.50).
        """
        m = mask.astype(np.float32)
        total = m.sum() + 1e-8
        if total < 10: return 0.5
        h, w = m.shape
        # eixo vertical: metade superior vs inferior espelhada
        top    = m[:h//2, :]
        bottom = np.flipud(m[h//2:, :])
        mh = min(top.shape[0], bottom.shape[0])
        asym_v = np.abs(top[:mh] - bottom[:mh]).sum() / total
        # eixo horizontal: metade esquerda vs direita espelhada
        left  = m[:, :w//2]
        right = np.fliplr(m[:, w//2:])
        mw = min(left.shape[1], right.shape[1])
        asym_h = np.abs(left[:, :mw] - right[:, :mw]).sum() / total
        return float(np.clip((asym_v + asym_h) / 2, 0, 1))

    def sB(self,cam,mask):
        mu8=mask.astype(np.uint8)
        border=(mu8-cv2.erode(mu8,np.ones((15,15),np.uint8),iterations=2)).astype(bool)
        tot=cam[mask].sum()+1e-8
        brd=cam[border].sum() if border.any() else 0.0
        return float(np.clip(brd/tot/0.30,0,1))

    def sC(self,cam,img,mask):
        pix=img[mask]
        if len(pix)<50: return 0.0
        pl=cv2.cvtColor(pix.reshape(-1,1,3),cv2.COLOR_RGB2Lab).reshape(-1,3).astype(np.float32)
        s=pl[np.random.choice(len(pl),min(800,len(pl)),replace=False)]
        bk=2; prev=None
        for k in range(2,6):
            km=KMeans(n_clusters=k,n_init=3,max_iter=50,random_state=42).fit(s)
            if prev and (prev-km.inertia_)/(prev+1e-8)<0.20: break
            bk=k; prev=km.inertia_
        kmf=KMeans(n_clusters=bk,n_init=5,max_iter=100,random_state=42).fit(pl)
        cf=cam[mask]
        heats=[cf[kmf.labels_==z].mean() for z in range(bk) if (kmf.labels_==z).sum()>0]
        if len(heats)<2: return 0.0
        return float(np.clip(np.std(heats)/0.3,0,1))

    def sD(self,cam,mask):
        lbp=local_binary_pattern((cam*255).astype(np.uint8),P=8,R=1,method='uniform')
        ll=lbp[mask]
        if len(ll)==0: return 0.0
        h,_=np.histogram(ll,bins=10,range=(0,10),density=True); h+=1e-10
        return float(np.clip(-np.sum(h*np.log2(h))/3.32,0,1))

    def extrair(self,path):
        img=cv2.cvtColor(cv2.imread(path),cv2.COLOR_BGR2RGB)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE))
        t=self.tfm(Image.fromarray(img)).unsqueeze(0).to(self.device)
        with torch.enable_grad():
            p=float(self.model(t.clone()).squeeze())
        cam=self.get_cam(t); mask=self.segment(img)
        return p,cam,img,mask,self.sA(mask),self.sB(cam,mask),self.sC(cam,img,mask),self.sD(cam,mask)  # sA agora usa mask

extrator = GradCAMExtractor(model, DEVICE)
print('GradCAMExtractor pronto')

## 6. `GradCAMExtractor` — extração das 4 features ABCD

Classe central do pipeline. Para cada imagem, executa, em sequência:

1. **`get_cam()`** — Grad-CAM padrão sobre a última camada convolucional
   (`features[-1]`, mapa 7×7×1280), upsampled para 224×224 e normalizado a
   `[0,1]`.
2. **`segment()`** — segmentação da lesão via Otsu (threshold inicial) +
   GrabCut (refinamento), com fallback silencioso para a máscara Otsu pura em
   caso de erro do GrabCut ou contorno inválido (bounding box < 10px).
3. **Quatro métricas ABCD**, cada uma com correspondência clínica explícita:
   - **`sA(mask)`** — assimetria geométrica real da máscara segmentada
     (compara metades espelhadas nos eixos vertical/horizontal). Esta é a
     correção mais importante da v3: a versão anterior usava
     `A = P(maligno) × 0.3` como proxy — redundante com a 5ª entrada do fuzzy
     (correlação ≈0.04 com o rótulo real, porque era basicamente um eco de P).
     A versão atual mede assimetria de fato, independente de P.
   - **`sB(cam, mask)`** — proporção da saliência Grad-CAM concentrada num
     anel de borda (~15px de erosão morfológica). Tem distribuição fortemente
     bimodal e é normalizada via `QuantileTransformer` antes de entrar no fuzzy.
   - **`sC(cam, img, mask)`** — dispersão da saliência média entre zonas de cor
     (K-Means em CIE L*a*b*, k escolhido por método do cotovelo, k∈[2,5]).
   - **`sD(cam, mask)`** — entropia de Shannon do histograma LBP do mapa
     Grad-CAM; proxy computável para "Differential Structures" (o D
     dermatoscópico de Nachbar et al. 1994, não o "Diameter" da mnemônica
     popular — ver discussão na Seção 3.4 do artigo).

> ⚠️ **Efeito colateral conhecido desta correção:** trocar o proxy de A pela
> assimetria real desacopla A de P. Como a rule base de 16 regras foi desenhada
> e calibrada pensando no proxy antigo (onde A e P eram quase redundantes),
> essa correção tende a reduzir a sensibilidade do sistema fuzzy em relação aos
> números publicados no artigo (que usam o proxy). Ver discussão completa nas
> notas de validação cruzada (Seção 9).


In [ ]:
# ══════════════════════════════════════════════════════════
# CALIBRAÇÃO — coleta A,B,C,D,P do conjunto de treino
# P (p_maligno) é a 5ª feature — entra diretamente no fuzzy
# ══════════════════════════════════════════════════════════
print(f'Calibrando com {len(df_train)} imagens de treino...')
Ac,Bc,Cc,Dc,Pc,yc=[],[],[],[],[],[]

for _,row in tqdm(df_train.iterrows(),total=len(df_train),desc='Calibração'):
    try:
        p,cam,img,mask,A,B,C,D=extrator.extrair(row['image_path'])
        Ac.append(A); Bc.append(B); Cc.append(C); Dc.append(D); Pc.append(p)
        yc.append(int(row['is_malignant']))
    except: pass

Ac=np.array(Ac); Bc=np.array(Bc); Cc=np.array(Cc)
Dc=np.array(Dc); Pc=np.array(Pc); yc=np.array(yc)
mal_c=(yc==1); ben_c=(yc==0)
print(f'Coletadas: {len(Ac)} | mal={mal_c.sum()} | ben={ben_c.sum()}')

# Diagnóstico de discriminação
print('\nCorrelações com malignidade:')
for nm,vals in [('A_raw',Ac),('B_raw',Bc),('C',Cc),('D',Dc),('P(rede)',Pc)]:
    diff=vals[mal_c].mean()-vals[ben_c].mean()
    r=np.corrcoef(vals,yc.astype(float))[0,1]
    print(f'  {nm:8s}: diff={diff:+.4f}  r={r:+.4f}')

# Normaliza B (resolve bimodalidade)
qt_B=QuantileTransformer(output_distribution='uniform',random_state=42)
qt_B.fit(Bc.reshape(-1,1))
Bc_n=qt_B.transform(Bc.reshape(-1,1)).ravel()

def calc_params_safe(vals,mal_m,ben_m,ql=25,qh=75):
    vmin=float(np.percentile(vals,2)); vmax=float(np.percentile(vals,98))
    span=vmax-vmin
    if span<1e-6: return vmin,vmax,vmin+span*.25,vmin+span*.50,vmin+span*.75
    mg=max(span*.02,1e-4)
    mc=float(np.clip(np.median(vals),vmin+mg,vmax-mg))
    bt=float(np.clip(np.percentile(vals[ben_m],ql),vmin+mg,mc-mg))
    ab=float(np.clip(np.percentile(vals[mal_m],qh),mc+mg,vmax-mg))
    if not(bt<mc<ab): bt=vmin+span*.25;mc=vmin+span*.50;ab=vmin+span*.75
    return vmin,vmax,bt,mc,ab

params_cal={}
for feat,vals in [('A',Ac),('C',Cc),('D',Dc),('P',Pc)]:
    params_cal[feat]=calc_params_safe(vals,mal_c,ben_c)
params_cal['B']=calc_params_safe(Bc_n,mal_c,ben_c)

print('\nParâmetros calibrados:')
for feat in ['A','B','C','D','P']:
    vmin,vmax,bt,mc,ab=params_cal[feat]
    ok=(bt<mc<ab)
    print(f'  {feat}: bt={bt:.4f} mc={mc:.4f} ab={ab:.4f} {"✓" if ok else "✗"}')

## 7. Calibração das funções de pertinência

Para cada variável `X ∈ {A, B, C, D, P}`, calcula três pontos de transição
**a partir do conjunto de treino apenas**:

- `bt` = percentil 25 de `X` em lesões benignas → limite superior de "baixo"
- `mc` = mediana global de `X` → centro de "médio"
- `ab` = percentil 75 de `X` em lesões malignas → limite inferior de "alto"

`calc_params_safe()` garante `bt < mc < ab`; se a calibração violar essa ordem
(distribuições muito sobrepostas), recorre a um fallback de tercis uniformes
sobre o range `[percentil 2, percentil 98]`.

A variável `B` é normalizada via `QuantileTransformer` (ajustado **somente no
treino**) antes da calibração, corrigindo sua distribuição bimodal bruta.

O bloco também imprime a correlação de Pearson de cada feature com o rótulo —
usada no artigo para justificar os pesos das camadas de regras (P tem a maior
correlação, r≈0.71; A/B/C/D ficam em uma faixa mais moderada).


In [ ]:
# FUZZY v3
# [COR1] mf(): overlap=span*0.25 elimina zonas mortas da v2 (tl/th deixavam buracos)
# [COR2] MFs saida: sobreposicao uniforme 1.5pt, sem lacunas em [0,10]
# [COR3] inferir_v3: Ai=A real do sA(cam), nao mais p_mal*0.3
# [COR4] fallback: pesos somam 10, resultado direto em [0,10] sem magic number

def build_fuzzy_v3(params):
    u=np.arange(0,1.001,0.002); r=np.arange(0,10.1,.1)
    A_=ctrl.Antecedent(u,'assim');   B_=ctrl.Antecedent(u,'borda')
    C_=ctrl.Antecedent(u,'cor');     D_=ctrl.Antecedent(u,'estrutura')
    P_=ctrl.Antecedent(u,'rede');    R_=ctrl.Consequent(r,'risco')

    def mf(ant,p):
        vmin,vmax,bt,mc,ab=p
        span=vmax-vmin
        overlap=max(span*0.25,1e-4)  # [COR1] sobreposicao explicita
        bt_low =float(np.clip(bt-overlap*0.5, vmin,        mc-1e-4))
        bt_high=float(np.clip(bt+overlap*0.5, bt_low+1e-4, mc))
        ab_low =float(np.clip(ab-overlap*0.5, mc,          vmax-1e-4))
        ab_high=float(np.clip(ab+overlap*0.5, ab_low+1e-4, vmax))
        ant['baixo']=fuzz.trapmf(u,[vmin,vmin,bt_low, bt_high])
        ant['medio']=fuzz.trimf (u,[bt_low,mc,ab_high])
        ant['alto'] =fuzz.trapmf(u,[ab_low,ab_high,vmax,vmax])

    for ant,k in [(A_,'A'),(B_,'B'),(C_,'C'),(D_,'D'),(P_,'P')]:
        mf(ant,params[k])

    # [COR2] sobreposicao uniforme 1.5pt em todos os termos adjacentes
    # v2: muito_baixo->2.5, baixo comecava em 1.5 mas nucleo em 3.0 = buraco
    # v2: alto->8.5, muito_alto so em 7.5 = sobreposicao torta
    R_['muito_baixo']=fuzz.trapmf(r,[0,  0,  1.5,3.0])
    R_['baixo']      =fuzz.trimf (r,[1.5,3.0,4.5])
    R_['medio']      =fuzz.trimf (r,[3.0,5.0,7.0])
    R_['alto']       =fuzz.trimf (r,[5.5,7.0,8.5])
    R_['muito_alto'] =fuzz.trapmf(r,[7.0,8.5,10, 10])

    rules=[
        ctrl.Rule(P_['alto'] &A_['alto'],               R_['muito_alto'],label='R1:P+A_altos'),
        ctrl.Rule(P_['alto'] &B_['alto'],               R_['muito_alto'],label='R2:P+B_altos'),
        ctrl.Rule(P_['alto'] &C_['alto'],               R_['alto'],      label='R3:P+C_altos'),
        ctrl.Rule(P_['alto'],                           R_['alto'],      label='R4:P_alto'),
        ctrl.Rule(P_['baixo']&A_['baixo'],              R_['muito_baixo'],label='R5:P+A_baixos'),
        ctrl.Rule(P_['baixo'],                          R_['baixo'],     label='R6:P_baixo'),
        ctrl.Rule(P_['medio']&A_['alto']&B_['alto'],    R_['alto'],      label='R7:Pm+A+B'),
        ctrl.Rule(P_['medio']&A_['alto']&C_['alto'],    R_['alto'],      label='R8:Pm+A+C'),
        ctrl.Rule(P_['medio']&B_['alto']&D_['alto'],    R_['alto'],      label='R9:Pm+B+D'),
        ctrl.Rule(P_['medio']&A_['alto'],               R_['medio'],     label='R10:Pm+A'),
        ctrl.Rule(P_['medio']&B_['alto'],               R_['medio'],     label='R11:Pm+B'),
        ctrl.Rule(P_['medio'],                          R_['medio'],     label='R12:Pm'),
        ctrl.Rule(P_['medio']&A_['baixo']&B_['baixo'],  R_['baixo'],     label='R13:Pm+A+B_bx'),
        ctrl.Rule(P_['medio']&A_['baixo']&C_['baixo'],  R_['baixo'],     label='R14:Pm+A+C_bx'),
        ctrl.Rule(A_['alto']&B_['alto']&C_['alto'],     R_['muito_alto'],label='R15:Triade_CAM'),
        ctrl.Rule(C_['alto']&D_['alto'],                R_['medio'],     label='R16:C+D'),
    ]
    return ctrl.ControlSystemSimulation(ctrl.ControlSystem(rules))


fuzzy_v3 = build_fuzzy_v3(params_cal)
print('Fuzzy v3 (16 regras, MFs corrigidas) OK')

def norm_B(b): return float(qt_B.transform([[float(b)]])[0,0])

def nivel_risco(s):
    if   s<2.5: return 'MUITO BAIXO'
    elif s<5.0: return 'BAIXO'
    elif s<7.0: return 'MODERADO'
    elif s<8.5: return 'ALTO'
    else:       return 'MUITO ALTO'

def inferir_v3(A,B,C,D,p_mal):
    Bn=norm_B(B)
    Ai=float(np.clip(A,    0,1))  # [COR3] A real do sA(cam), nao mais p_mal*0.3
    Pi=float(np.clip(p_mal,0,1))
    try:
        fuzzy_v3.input['assim']    =Ai
        fuzzy_v3.input['borda']    =float(np.clip(Bn,0,1))
        fuzzy_v3.input['cor']      =float(np.clip(C, 0,1))
        fuzzy_v3.input['estrutura']=float(np.clip(D, 0,1))
        fuzzy_v3.input['rede']     =Pi
        fuzzy_v3.compute()
        return float(fuzzy_v3.output['risco']),Bn,Ai,Pi
    except:
        s=float(np.clip(Pi*5.0+Ai*2.0+Bn*1.2+C*1.0+D*0.8,0,10))  # [COR4] pesos somam 10
        return s,Bn,Ai,Pi

def interp_v3(s,Ai,Bn,C,D,Pi):
    p=params_cal
    pts=[
        f'P(rede)={Pi:.2f}->{"ALTO" if Pi>p["P"][4] else "medio" if Pi>p["P"][3] else "baixo"}',
        'ALTA ASSIMETRIA'       if Ai>p['A'][4] else 'assim. moderada' if Ai>p['A'][3] else 'simetrica',
        'BORDA IRREG.'          if Bn>p['B'][4] else 'borda leve.irreg' if Bn>p['B'][3] else 'borda regular',
        'PIGMENT. VAR.'         if C >p['C'][4] else 'pigment. variavel' if C >p['C'][3] else 'pigment. uniforme',
        'ESTRUTURA COMPL.'      if D >p['D'][4] else 'estrutura moder.' if D >p['D'][3] else 'estrutura simples',
    ]
    concl={'MUITO BAIXO':'monitoramento anual','BAIXO':'avaliacao em 6 meses',
           'MODERADO':'avaliacao dermatologica','ALTO':'biopsia recomendada',
           'MUITO ALTO':'encaminhamento urgente'}[nivel_risco(s)]
    return f"{' | '.join(pts)} --- {concl}"

def regras_v3(Ai,Bn,C,D,Pi):
    p=params_cal; rs=[]
    if Pi>p['P'][4] and Ai>p['A'][4]:                          rs.append('R1:P+A')
    if Pi>p['P'][4] and Bn>p['B'][4]:                          rs.append('R2:P+B')
    if Pi>p['P'][4] and C>p['C'][4]:                           rs.append('R3:P+C')
    if Pi>p['P'][4]:                                            rs.append('R4:P_alto')
    if Pi<p['P'][2] and Ai<p['A'][2]:                          rs.append('R5:P+A_bx')
    if Pi<p['P'][2]:                                            rs.append('R6:P_baixo')
    if p['P'][2]<Pi<p['P'][4] and Ai>p['A'][4] and Bn>p['B'][4]: rs.append('R7:Pm+A+B')
    if Ai>p['A'][4] and Bn>p['B'][4] and C>p['C'][4]:         rs.append('R15:Triade')
    if not rs: rs.append('intermediarias')
    return ' | '.join(rs)

print('Funcoes auxiliares OK')

## 8. Sistema Fuzzy Mamdani — construção e funções de inferência

`build_fuzzy_v3(params)` constrói o sistema completo:

- **5 antecedentes** (`assim`, `borda`, `cor`, `estrutura`, `rede`), cada um
  com 3 termos linguísticos (baixo/médio/alto), funções trapezoidal–
  triangular–trapezoidal com 25% de sobreposição entre termos adjacentes.
- **1 consequente** (`risco`, domínio `[0,10]`) com 5 termos, sobreposição
  uniforme de 1.5pt entre termos adjacentes (correção da v2, que tinha
  lacunas/sobreposições inconsistentes).
- **16 regras** organizadas em 3 camadas hierárquicas:
  - **R1–R6**: `P` (confiança da rede) alto/baixo → tratado como autoritativo
  - **R7–R14**: `P` médio (rede incerta) → decisão transferida para A/B/C/D
  - **R15–R16**: padrões dermatoscópicos clássicos (tríade A+B+C; co-ocorrência
    C+D)

`inferir_v3()` executa a inferência para uma amostra, com fallback linear
(pesos somando 10) caso o solver fuzzy falhe. `regras_v3()` e `interp_v3()`
geram, respectivamente, a lista de regras disparadas e uma explicação textual
em linguagem natural — usadas no relatório XLSX final (Seção 12).


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CROSS-VALIDATION ESTRATIFICADA — 5 folds no conjunto de TREINO
#
# Pipeline por fold:
#   1. Reutiliza arrays Ac,Bc,Cc,Dc,Pc,yc extraídos na calibração (cell 7)
#      → não re-executa Grad-CAM; apenas re-calibra o fuzzy por fold
#   2. QuantileTransformer(B) e params das MFs calibrados APENAS no
#      fold de treino — sem leakage para o fold de validação
#   3. Threshold Youden calculado dentro de cada fold de validação
#
# ⚠ O modelo EfficientNet NÃO é re-treinado: pesos fixos em todos os folds.
# ══════════════════════════════════════════════════════════════════════
from sklearn.model_selection import StratifiedKFold

N_FOLDS = 5
skf     = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

fold_metrics_net = []
fold_metrics_fuz = []
fold_aucs        = []

print(f'Cross-validation {N_FOLDS}-fold estratificada | treino: {len(Ac)} amostras')
print(f'{"Fold":>5} | {"AUC-Net":>8} | {"AUC-Fuz":>8} | '
      f'{"Acc-N":>7} | {"Acc-F":>7} | {"Sens-F":>7} | {"Spec-F":>7} | {"F1-F":>7}')
print('-' * 70)

for fold, (train_idx, val_idx) in enumerate(skf.split(Ac, yc), 1):

    # 1. Split dos arrays de features
    Atr, Aval = Ac[train_idx], Ac[val_idx]
    Btr, Bval = Bc[train_idx], Bc[val_idx]
    Ctr, Cval = Cc[train_idx], Cc[val_idx]
    Dtr, Dval = Dc[train_idx], Dc[val_idx]
    Ptr, Pval = Pc[train_idx], Pc[val_idx]
    ytr, yval = yc[train_idx], yc[val_idx]
    mal_f = (ytr == 1); ben_f = (ytr == 0)

    # 2. QuantileTransformer calibrado APENAS no fold de treino
    qt_fold = QuantileTransformer(output_distribution='uniform', random_state=42)
    qt_fold.fit(Btr.reshape(-1, 1))
    Btr_n  = qt_fold.transform(Btr.reshape(-1, 1)).ravel()
    Bval_n = qt_fold.transform(Bval.reshape(-1, 1)).ravel()

    # 3. Parâmetros das funções de pertinência — fold de treino
    params_fold = {}
    for feat, vals in [('A', Atr), ('C', Ctr), ('D', Dtr), ('P', Ptr)]:
        params_fold[feat] = calc_params_safe(vals, mal_f, ben_f)
    params_fold['B'] = calc_params_safe(Btr_n, mal_f, ben_f)

    # 4. Fuzzy calibrado neste fold
    fuzzy_fold = build_fuzzy_v3(params_fold)  # [COR5]

    def _norm_B_fold(b):
        return float(qt_fold.transform([[float(b)]])[0, 0])

    def _inferir_fold(A, B, C, D, p_mal):
        Bn = _norm_B_fold(B)
        Ai = float(np.clip(A, 0, 1))  # [COR5] A real
        Pi = float(np.clip(p_mal, 0, 1))
        try:
            fuzzy_fold.input['assim']     = Ai
            fuzzy_fold.input['borda']     = float(np.clip(Bn, 0, 1))
            fuzzy_fold.input['cor']       = float(np.clip(C,  0, 1))
            fuzzy_fold.input['estrutura'] = float(np.clip(D,  0, 1))
            fuzzy_fold.input['rede']      = Pi
            fuzzy_fold.compute()
            return float(fuzzy_fold.output['risco']), Bn, Ai, Pi
        except:
            # correto — linha única
            s = float(np.clip(Pi*5.0 + Ai*2.0 + Bn*1.2 + C*1.0 + D*0.8, 0, 10))
            return s, Bn, Ai, Pi

    # 5. Inferência no fold de validação
    scores_net, scores_fuz = [], []
    for i in range(len(val_idx)):
        scores_net.append(float(Pval[i]))
        s_f, _, _, _ = _inferir_fold(Aval[i], Bval[i], Cval[i], Dval[i], Pval[i])
        scores_fuz.append(s_f / 10.0)
    scores_net = np.array(scores_net)
    scores_fuz = np.array(scores_fuz)

    # 6. Métricas — threshold Youden calculado dentro do fold de validação
    auc_n = roc_auc_score(yval, scores_net)
    auc_f = roc_auc_score(yval, scores_fuz)
    fpr_n, tpr_n, thr_n_ = roc_curve(yval, scores_net)
    fpr_f, tpr_f, thr_f_ = roc_curve(yval, scores_fuz)
    t_n = float(thr_n_[np.argmax(tpr_n - fpr_n)])
    t_f = float(thr_f_[np.argmax(tpr_f - fpr_f)])

    def _met(yt, yp):
        tn, fp, fn, tp = confusion_matrix(yt, yp).ravel()
        return dict(acc=(tp+tn)/(tp+tn+fp+fn),
                    sens=tp/(tp+fn+1e-8),
                    spec=tn/(tn+fp+1e-8),
                    f1=2*tp/(2*tp+fp+fn+1e-8))

    mn_f = _met(yval, (scores_net >= t_n).astype(int))
    mf_f = _met(yval, (scores_fuz >= t_f).astype(int))
    fold_metrics_net.append(mn_f); fold_metrics_fuz.append(mf_f)
    fold_aucs.append([auc_n, auc_f])

    print(f'{fold:>5} | {auc_n:>8.4f} | {auc_f:>8.4f} | '
          f'{mn_f["acc"]:>7.3f} | {mf_f["acc"]:>7.3f} | '
          f'{mf_f["sens"]:>7.3f} | {mf_f["spec"]:>7.3f} | {mf_f["f1"]:>7.3f}')

# Resumo
aucs   = np.array(fold_aucs)
mn_arr = {k: np.array([m[k] for m in fold_metrics_net]) for k in ['acc','sens','spec','f1']}
mf_arr = {k: np.array([m[k] for m in fold_metrics_fuz]) for k in ['acc','sens','spec','f1']}

print('=' * 70)
print(f'{"MÉDIA":>5} | {aucs[:,0].mean():>8.4f} | {aucs[:,1].mean():>8.4f} | '
      f'{mn_arr["acc"].mean():>7.3f} | {mf_arr["acc"].mean():>7.3f} | '
      f'{mf_arr["sens"].mean():>7.3f} | {mf_arr["spec"].mean():>7.3f} | '
      f'{mf_arr["f1"].mean():>7.3f}')
print(f'{"±DP":>5} | {aucs[:,0].std():>8.4f} | {aucs[:,1].std():>8.4f} | '
      f'{mn_arr["acc"].std():>7.3f} | {mf_arr["acc"].std():>7.3f} | '
      f'{mf_arr["sens"].std():>7.3f} | {mf_arr["spec"].std():>7.3f} | '
      f'{mf_arr["f1"].std():>7.3f}')

# Dicionário para referenciar nos plots e no artigo
cv_summary = {
    'auc_net_mean': aucs[:,0].mean(), 'auc_net_std': aucs[:,0].std(),
    'auc_fuz_mean': aucs[:,1].mean(), 'auc_fuz_std': aucs[:,1].std(),
    'acc_fuz_mean': mf_arr['acc'].mean(), 'acc_fuz_std': mf_arr['acc'].std(),
    'sens_fuz_mean': mf_arr['sens'].mean(), 'sens_fuz_std': mf_arr['sens'].std(),
    'spec_fuz_mean': mf_arr['spec'].mean(), 'spec_fuz_std': mf_arr['spec'].std(),
    'f1_fuz_mean':  mf_arr['f1'].mean(),  'f1_fuz_std':  mf_arr['f1'].std(),
}
print()
print('cv_summary gerado:')
print(f"  AUC  fuzzy : {cv_summary['auc_fuz_mean']:.4f} \u00b1 {cv_summary['auc_fuz_std']:.4f}")
print(f"  Sens fuzzy : {cv_summary['sens_fuz_mean']:.3f} \u00b1 {cv_summary['sens_fuz_std']:.3f}")
print(f"  Spec fuzzy : {cv_summary['spec_fuz_mean']:.3f} \u00b1 {cv_summary['spec_fuz_std']:.3f}")
print(f"  F1   fuzzy : {cv_summary['f1_fuz_mean']:.3f} \u00b1 {cv_summary['f1_fuz_std']:.3f}")

# ── Plot: AUC por fold | métricas fuzzy | boxplot AUC ────────────────
fig_cv, axes = plt.subplots(1, 3, figsize=(15, 4))
fig_cv.suptitle(f'Cross-Validation {N_FOLDS}-fold — Fuzzy v3',
                fontsize=12, fontweight='bold')

# Painel 1: AUC por fold
ax = axes[0]
folds_x = np.arange(1, N_FOLDS + 1)
ax.plot(folds_x, aucs[:,0], 'o-', color='#3498DB', lw=2, label=f'Rede  {aucs[:,0].mean():.3f}')
ax.plot(folds_x, aucs[:,1], 's-', color='#27AE60', lw=2, label=f'Fuzzy {aucs[:,1].mean():.3f}')
ax.axhline(aucs[:,0].mean(), color='#3498DB', lw=1, ls='--', alpha=0.5)
ax.axhline(aucs[:,1].mean(), color='#27AE60', lw=1, ls='--', alpha=0.5)
ax.fill_between(folds_x,
                aucs[:,1] - aucs[:,1].std(),
                aucs[:,1] + aucs[:,1].std(),
                alpha=0.12, color='#27AE60')
ax.set_xlabel('Fold'); ax.set_ylabel('AUC-ROC')
ax.set_title('AUC por fold'); ax.legend(fontsize=9)
ax.grid(alpha=0.2); ax.set_xticks(folds_x)

# Painel 2: métricas fuzzy com barras de erro
ax = axes[1]
metric_keys  = ['acc', 'sens', 'spec', 'f1']
metric_names = ['Acc', 'Sens', 'Spec', 'F1']
means = [mf_arr[k].mean() for k in metric_keys]
stds  = [mf_arr[k].std()  for k in metric_keys]
colors_bar = ['#3498DB', '#E74C3C', '#27AE60', '#9B59B6']
bars = ax.bar(metric_names, means, color=colors_bar, edgecolor='white',
              yerr=stds, capsize=5, ecolor='#555555', error_kw={'lw': 1.2})
for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 0.005,
            f'{m:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylim(0, 1.15); ax.set_title('Métricas fuzzy (média ± DP)')
ax.grid(alpha=0.2, axis='y')

# Painel 3: boxplot AUC rede vs fuzzy
ax = axes[2]
bp = ax.boxplot([aucs[:,0], aucs[:,1]],
                labels=['Rede', 'Fuzzy v3'],
                patch_artist=True,
                medianprops=dict(color='#E74C3C', lw=2))
bp['boxes'][0].set_facecolor('#D6EAF8')
bp['boxes'][1].set_facecolor('#D5F5E3')
ax.set_title('Distribuição AUC (5 folds)')
ax.set_ylabel('AUC-ROC'); ax.grid(alpha=0.2, axis='y')

plt.tight_layout()
plt.savefig('/kaggle/working/cv_results_v3.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: /kaggle/working/cv_results_v3.png')


## 9. Validação cruzada 5-fold estratificada (sobre o treino)

Reaproveita os arrays `Ac, Bc, Cc, Dc, Pc, yc` já extraídos na calibração
(Seção 7) — **não roda o Grad-CAM de novo por fold**, apenas recalibra o fuzzy.

Pontos importantes de design:
- `QuantileTransformer` e os parâmetros das MFs são recalibrados **dentro de
  cada fold de treino**, sem leakage para o fold de validação.
- O threshold de decisão (Youden) é recalculado dentro de cada fold de
  validação — não é um valor fixo importado do conjunto de teste completo.
- **A rede EfficientNet não é re-treinada em nenhum fold** — pesos fixos,
  carregados uma única vez (Seção 5); apenas a camada fuzzy é recalibrada e
  revalidada fold a fold.

Gera `cv_summary` (médias ± desvio-padrão de AUC, acurácia, sensibilidade,
especificidade e F1 da camada fuzzy) e a figura `cv_results_v3.png` com 3
painéis: AUC por fold, métricas fuzzy com barras de erro, e boxplot comparativo
rede vs. fuzzy.

> 📊 **Achado relevante:** em execuções de CV, o AUC da rede se manteve estável
> em torno de 0.93 (consistente com o artigo), mas a sensibilidade média do
> fuzzy ficou bem abaixo dos 93.8% reportados no artigo — entre 67–68% em
> rodadas recentes com a `sA` real. Isso reforça a hipótese de que a rule base
> precisa de recalibração após a correção do critério A (ver nota na Seção 6).


In [ ]:
DX_MAP={'mel':'Melanoma','nv':'Nevo Melanocítico','bcc':'Carcinoma Basocelular',
        'akiec':'Ceratose Actínica','bkl':'Ceratose Seborreica',
        'df':'Dermatofibroma','vasc':'Lesão Vascular'}

print(f'Processando {len(df_test)} imagens...')
recs=[]; erros=0

for _,row in tqdm(df_test.iterrows(),total=len(df_test)):
    try:
        p,cam,img,mask,A,B,C,D=extrator.extrair(row['image_path'])
        s,Bn,Ai,Pi=inferir_v3(A,B,C,D,p)  # [COR6]
        nv=nivel_risco(s)
        recs.append({
            'image_id':    row['image_id'],
            'dx':          row['dx'],
            'dx_nome':     DX_MAP.get(row['dx'],row['dx']),
            'is_malignant':row['is_malignant'],
            'p_maligno':   round(p,4),
            'P_fuzzy':     round(Pi,4),
            'A_real':      round(A,4),  # [COR6] renomeado
            'A_usado':     round(Ai,4),
            'B_raw':       round(B,4),
            'B_norm':      round(Bn,4),
            'C_gradcam':   round(C,4),
            'D_gradcam':   round(D,4),
            'fuzzy_score': round(s,2),
            'nivel_risco': nv,
            'regras':      regras_v3(Ai,Bn,C,D,Pi),  # [COR6]
            'interpretacao': interp_v3(s,Ai,Bn,C,D,Pi),  # [COR6]
        })
    except: erros+=1

df_res=pd.DataFrame(recs)
y_tr=df_res['is_malignant'].values
y_pn=df_res['p_maligno'].values
y_pf=df_res['fuzzy_score'].values/10.0

auc_net=roc_auc_score(y_tr,y_pn)
auc_fuz=roc_auc_score(y_tr,y_pf)
fpr_n,tpr_n,thr_n=roc_curve(y_tr,y_pn)
fpr_f,tpr_f,thr_f=roc_curve(y_tr,y_pf)
thr_net2=float(thr_n[np.argmax(tpr_n-fpr_n)])
thr_fuz2=float(thr_f[np.argmax(tpr_f-fpr_f)])
y_pn2=(y_pn>=thr_net2).astype(int)
y_pf2=(y_pf>=thr_fuz2).astype(int)

def met(yt,yp):
    tn,fp,fn,tp=confusion_matrix(yt,yp).ravel()
    return dict(acc=(tp+tn)/(tp+tn+fp+fn),sens=tp/(tp+fn+1e-8),
                spec=tn/(tn+fp+1e-8),f1=2*tp/(2*tp+fp+fn+1e-8),
                tp=tp,fp=fp,fn=fn,tn=tn)

mn=met(y_tr,y_pn2); mf=met(y_tr,y_pf2)

print(f'\n{"="*58}')
print(f'  RESULTADOS — FUZZY v3 (5 entradas + P(rede))')
print(f'  Comparação vs v1 (proxy antigo, valores fixos de referência):')
print(f'  AUC fuzzy anterior : 0.6478  →  novo: {auc_fuz:.4f}')
print(f'  Spec anterior      : 47.6%   →  novo: {mf["spec"]*100:.1f}%')
print(f'  Acc anterior       : 68.7%   →  novo: {mf["acc"]*100:.1f}%')
print(f'  F1 anterior        : 0.7493  →  novo: {mf["f1"]:.4f}')
print()
print(f'  AUC rede  : {auc_net:.4f}')
print(f'  AUC fuzzy : {auc_fuz:.4f}')
print(f'  REDE : Acc={mn["acc"]:.3f} Sens={mn["sens"]:.3f} Spec={mn["spec"]:.3f} F1={mn["f1"]:.3f}')
print(f'  FUZZY: Acc={mf["acc"]:.3f} Sens={mf["sens"]:.3f} Spec={mf["spec"]:.3f} F1={mf["f1"]:.3f}')
print(f'  VP={mf["tp"]} FP={mf["fp"]} FN={mf["fn"]} VN={mf["tn"]}')
print()
print('  Distribuição de risco:')
mal_r=(y_tr==1); ben_r=(y_tr==0)
for nv in ['MUITO BAIXO','BAIXO','MODERADO','ALTO','MUITO ALTO']:
    n_nv=  (df_res['nivel_risco']==nv).sum()
    nm_nv= ((df_res['nivel_risco']==nv)&mal_r).sum()
    nb_nv= ((df_res['nivel_risco']==nv)&ben_r).sum()
    print(f'  {nv:12s}: total={n_nv:3d} | mal={nm_nv:3d} | ben={nb_nv:3d}')
print()
print(classification_report(y_tr,y_pf2,target_names=['Benigno','Maligno']))

## 10. Avaliação no conjunto de teste held-out

Roda `extrator.extrair()` + `inferir_v3()` para cada imagem de `df_test`,
monta `df_res` com todas as features, score fuzzy, nível de risco, regras
disparadas e interpretação textual. O threshold de decisão (Youden) é
calculado separadamente para a saída da rede e para a saída fuzzy.

> ℹ️ Os valores impressos como "v1 (proxy antigo)" são uma referência fixa
> escrita no código-fonte (não recalculados a partir de uma execução anterior
> salva). Se a rule base ou os dados de calibração mudarem novamente,
> atualize esses valores manualmente para manter a comparação válida.


In [ ]:
fig=plt.figure(figsize=(18,12))
fig.suptitle(
    f'Fuzzy v3 — 5 entradas (A,B,C,D,P) | {MAX_IMAGES} imgs\n'
    f'AUC rede={auc_net:.3f}  AUC fuzzy={auc_fuz:.3f}  '
    f'Acc={mf["acc"]*100:.1f}%  Sens={mf["sens"]*100:.1f}%  Spec={mf["spec"]*100:.1f}%',
    fontsize=12,fontweight='bold')
gs=gridspec.GridSpec(2,3,hspace=0.45,wspace=0.35)

ax1=fig.add_subplot(gs[0,0])
ConfusionMatrixDisplay(confusion_matrix(y_tr,y_pf2),
    display_labels=['Benigno','Maligno']).plot(ax=ax1,colorbar=False,cmap='Blues')
ax1.set_title('Confusão — Fuzzy v3',fontweight='bold')

ax2=fig.add_subplot(gs[0,1])
ax2.plot(fpr_n,tpr_n,color='#3498DB',lw=2,label=f'Rede {auc_net:.3f}')
ax2.plot(fpr_f,tpr_f,color='#27AE60',lw=2,label=f'Fuzzy v3 {auc_fuz:.3f}')
# Mostra curva anterior como referência
ax2.axhline(0.6478,color='#E74C3C',linestyle=':',lw=1,alpha=0.5,label='Fuzzy v1 0.648')
ax2.plot([0,1],[0,1],'k--',lw=1,alpha=0.3)
ax2.set_title('ROC — v1 vs v2',fontweight='bold')
ax2.legend(fontsize=8); ax2.grid(alpha=0.2)

ax3=fig.add_subplot(gs[0,2])
ax3.hist(df_res['fuzzy_score'][ben_r],bins=20,alpha=0.6,color='#27AE60',
         label='Benigno',edgecolor='white',density=True)
ax3.hist(df_res['fuzzy_score'][mal_r],bins=20,alpha=0.6,color='#E74C3C',
         label='Maligno',edgecolor='white',density=True)
ax3.axvline(thr_fuz2*10,color='black',linestyle='--',lw=1.5,label=f'thr={thr_fuz2*10:.1f}')
ax3.set_title('Distribuição Fuzzy v3',fontweight='bold')
ax3.legend(fontsize=8); ax3.grid(alpha=0.2)

ax4=fig.add_subplot(gs[1,0])
feats4=['p_maligno','A_real','B_norm','C_gradcam','D_gradcam']
lbs4  =['P(rede)','A real','B norm','C cor','D estrut']
mm4=[df_res[f][mal_r].mean() for f in feats4]
mb4=[df_res[f][ben_r].mean() for f in feats4]
x4=np.arange(5); w4=0.35
ax4.bar(x4-w4/2,mm4,w4,color='#E74C3C',label='Maligno',edgecolor='white')
ax4.bar(x4+w4/2,mb4,w4,color='#27AE60',label='Benigno',edgecolor='white')
ax4.set_xticks(x4); ax4.set_xticklabels(lbs4,fontsize=8)
ax4.set_title('Separação por feature',fontweight='bold')
ax4.legend(fontsize=8); ax4.grid(alpha=0.2,axis='y')

ax5=fig.add_subplot(gs[1,1])
nvs=['MUITO BAIXO','BAIXO','MODERADO','ALTO','MUITO ALTO']
nm=[((df_res['nivel_risco']==v)&mal_r).sum() for v in nvs]
nb_=[((df_res['nivel_risco']==v)&ben_r).sum() for v in nvs]
x5=np.arange(5); w5=0.4
ax5.bar(x5-w5/2,nm,w5,color='#E74C3C',label='Maligno',edgecolor='white')
ax5.bar(x5+w5/2,nb_,w5,color='#27AE60',label='Benigno',edgecolor='white')
ax5.set_xticks(x5); ax5.set_xticklabels(['M.Bx','Baixo','Mod','Alto','M.Al'],fontsize=8)
ax5.set_title('Risco por classe',fontweight='bold')
ax5.legend(fontsize=8); ax5.grid(alpha=0.2,axis='y')

ax6=fig.add_subplot(gs[1,2]); ax6.axis('off')
ax6.text(0.05,0.95,
    f'COMPARAÇÃO v1 → v2:\n\n'
    f'AUC  : 0.6478 → {auc_fuz:.4f}\n'
    f'Acc  : 68.7%  → {mf["acc"]*100:.1f}%\n'
    f'Spec : 47.6%  → {mf["spec"]*100:.1f}%\n'
    f'F1   : 0.749  → {mf["f1"]:.3f}\n\n'
    f'5 ENTRADAS FUZZY:\n'
    f'P(rede) — dominante\n'
    f'  bt={params_cal["P"][2]:.3f} ab={params_cal["P"][4]:.3f}\n'
    f'A real (sA da mascara)\n'
    f'  bt={params_cal["A"][2]:.3f} ab={params_cal["A"][4]:.3f}\n'
    f'B borda (rank)\n'
    f'  bt={params_cal["B"][2]:.3f} ab={params_cal["B"][4]:.3f}\n'
    f'C cor / D estrut (apoio)\n\n'
    f'16 regras | thr={thr_fuz2*10:.2f}/10',
    transform=ax6.transAxes,fontsize=9,verticalalignment='top',fontfamily='monospace',
    bbox=dict(boxstyle='round',facecolor='#EBF5FB',alpha=0.8))
plt.savefig('/kaggle/working/dashboard_fuzzy_v3.png',dpi=150,bbox_inches='tight',facecolor='white')
plt.show()
print('Salvo: /kaggle/working/dashboard_fuzzy_v3.png')

## 11. Dashboard de visualização

6 painéis: matriz de confusão, curva ROC (rede vs. fuzzy, com a v1 marcada como
referência horizontal), histograma de distribuição do score fuzzy por classe,
comparação de médias por feature (maligno vs. benigno), distribuição de níveis
de risco por classe, e um painel de texto-resumo com os parâmetros de
calibração das entradas dominantes. Salvo em `dashboard_fuzzy_v3.png`.


In [ ]:
def fill(h): return PatternFill('solid',fgColor=h)
def bdr():
    t=Side(border_style='thin',color='CCCCCC'); return Border(left=t,right=t,top=t,bottom=t)
def aln(h='center',v='center',w=True): return Alignment(horizontal=h,vertical=v,wrap_text=w)
def rfill(s):
    if s<2.5: return fill('CCFFCC'),Font(color='005000',bold=True,size=9)
    elif s<4.5: return fill('C6EFCE'),Font(color='276221',bold=True,size=9)
    elif s<6.0: return fill('FFEB9C'),Font(color='9C5700',bold=True,size=9)
    elif s<7.5: return fill('FFC7CE'),Font(color='9C0006',bold=True,size=9)
    else: return fill('FF6666'),Font(color='7B0000',bold=True,size=9)

wb=openpyxl.Workbook()
ws=wb.active; ws.title='Resultados'; ws.freeze_panes='A3'
ws.merge_cells('A1:S1')
ws['A1'].value=(f'FUZZY v3 — 5 ENTRADAS (A,B,C,D,P) | {MAX_IMAGES}imgs '
                f'| AUC rede={auc_net:.3f} | AUC fuzzy={auc_fuz:.3f}')
ws['A1'].font=Font(bold=True,size=11,color='FFFFFF')
ws['A1'].fill=fill('1B3A5C'); ws['A1'].alignment=aln(); ws.row_dimensions[1].height=26

cols=[
    ('ID Imagem',14),('Diagnóstico',22),('É Câncer?',13),
    ('P(Maligno)\nRede',13),('P fuzzy\n(entrada)',12),
    ('A real (sA CAM)',10),('A usado',12),
    ('B raw',10),('B norm',12),('C',11),('D',12),
    ('Score\nFuzzy',12),('Nível\nRisco',15),('Predição',12),
    ('Interpretação Clínica',52),('Regras Ativadas',34),
    ('Acerto\nRede',11),('Acerto\nFuzzy',11),('Concorda',10),
]
for ci,(nm,larg) in enumerate(cols,1):
    c=ws.cell(row=2,column=ci,value=nm)
    c.font=Font(bold=True,color='FFFFFF',size=8); c.fill=fill('1F4E79')
    c.border=bdr(); c.alignment=aln()
    ws.column_dimensions[get_column_letter(ci)].width=larg
ws.row_dimensions[2].height=36

for ri,row in df_res.iterrows():
    r=ri+3; canc=bool(row['is_malignant']); s=row['fuzzy_score']; p=row['p_maligno']
    pn=p>=thr_net2; pf=s/10>=thr_fuz2
    an='✓' if pn==canc else '✗'; af='✓' if pf==canc else '✗'; cc='✓' if pn==pf else '✗'
    vals=[
        row['image_id'],row['dx_nome'],
        '✓ MALIGNO' if canc else '✗ BENIGNO',
        round(p,4),round(row['P_fuzzy'],4),
        row['A_real'],row['A_usado'],row['B_raw'],row['B_norm'],
        row['C_gradcam'],row['D_gradcam'],
        round(s,2),row['nivel_risco'],
        'MALIGNO' if pf else 'BENIGNO',
        row['interpretacao'],row['regras'],an,af,cc]
    for ci,val in enumerate(vals,1):
        c=ws.cell(row=r,column=ci,value=val); c.border=bdr(); c.font=Font(size=8)
        c.alignment=aln(h='center' if ci in [1,3,4,5,6,7,8,9,10,11,12,17,18,19] else 'left')
        if ci==3:
            c.fill=fill('FFC7CE') if canc else fill('C6EFCE')
            c.font=Font(bold=True,size=8,color='9C0006' if canc else '276221')
        if ci==12: rf,rft=rfill(s); c.fill=rf; c.font=rft; c.number_format='0.00'
        if ci==13: rf,rft=rfill(s); c.fill=rf; c.font=rft
        if ci in [17,18]:
            ok=(val=='✓'); c.fill=fill('C6EFCE') if ok else fill('FFC7CE')
            c.font=Font(bold=True,size=10,color='276221' if ok else '9C0006'); c.alignment=aln()
        if ci==19:
            ok=(val=='✓'); c.fill=fill('EEF2F7') if ok else fill('FFF0CC')
            c.font=Font(size=9,bold=True,color='1B3A5C' if ok else '7B3800')
    ws.row_dimensions[r].height=24

# ABA 2 — Métricas
ws2=wb.create_sheet('Métricas')
for col,w in zip('ABCDE',[28,14,14,14,16]): ws2.column_dimensions[col].width=w
ws2.merge_cells('A1:E1'); ws2['A1'].value='COMPARAÇÃO DE MÉTRICAS'
ws2['A1'].font=Font(bold=True,size=12,color='FFFFFF')
ws2['A1'].fill=fill('1B3A5C'); ws2['A1'].alignment=aln()
for ci,h in enumerate(['Métrica','Baseline','Sol.B pura','Fuzzy v1','Fuzzy v2'],1):
    c=ws2.cell(row=2,column=ci,value=h); c.font=Font(bold=True,color='FFFFFF',size=9)
    c.fill=fill('1F4E79'); c.border=bdr(); c.alignment=aln()
tab=[
    ('AUC-ROC',       '0.530','0.927','0.648',f'{auc_fuz:.4f}'),
    ('Acurácia',      '56.9%','84.8%','68.7%',f'{mf["acc"]*100:.1f}%'),
    ('Sensibilidade', '45.1%','88.1%','86.5%',f'{mf["sens"]*100:.1f}%'),
    ('Especificidade','64.9%','82.6%','47.6%',f'{mf["spec"]*100:.1f}%'),
    ('F1',            '0.459','0.825','0.749',f'{mf["f1"]:.4f}'),
    ('FP (benignos→mal)','—','—','144',str(mf['fp'])),
    ('FN (malignos→ben)','—','—','44', str(mf['fn'])),
    ('N entradas fuzzy','4','—','4','5 (+P rede)'),
    ('N regras',      '12','—','12','16'),
]
for ri4,rd in enumerate(tab,3):
    for ci,val in enumerate(rd,1):
        c=ws2.cell(row=ri4,column=ci,value=val)
        c.border=bdr(); c.alignment=aln(); c.font=Font(size=9)
        if ri4%2==0: c.fill=fill('F0F4F8')
        if ci==5: c.fill=fill('D5F5E3')

# ABA 3 — Regras
ws3=wb.create_sheet('Regras Fuzzy v3')
for col,w in zip('ABCD',[8,52,28,40]): ws3.column_dimensions[col].width=w
ws3.merge_cells('A1:D1')
ws3['A1'].value='BASE DE REGRAS — FUZZY v3 (5 entradas, P domina)'
ws3['A1'].font=Font(bold=True,size=11,color='FFFFFF')
ws3['A1'].fill=fill('1B3A5C'); ws3['A1'].alignment=aln()
for ci,h in enumerate(['#','Condição SE...','ENTÃO...','Lógica clínica'],1):
    c=ws3.cell(row=2,column=ci,value=h)
    c.font=Font(bold=True,color='FFFFFF',size=9)
    c.fill=fill('1F4E79'); c.border=bdr(); c.alignment=aln()
regras_doc=[
    ('R1','P(rede) ALTO e Assimetria ALTA',            'Risco MUITO ALTO','Rede + CAM confirmam maligno'),
    ('R2','P(rede) ALTO e Borda IRREGULAR',            'Risco MUITO ALTO','Rede + borda suspeita'),
    ('R3','P(rede) ALTO e Cor VARIÁVEL',               'Risco ALTO',      'Rede + heterogeneidade cromática'),
    ('R4','P(rede) ALTO (sozinho)',                    'Risco ALTO',      'Rede detecta maligno com alta confiança'),
    ('R5','P(rede) BAIXO e Assimetria BAIXA',          'Risco MUITO BAIXO','Rede + CAM confirmam benigno'),
    ('R6','P(rede) BAIXO (sozinho)',                   'Risco BAIXO',     'Rede descarta malignidade'),
    ('R7','P médio + Assimetria ALTA + Borda IRREG',   'Risco ALTO',      'CAM indica suspeita quando rede incerta'),
    ('R8','P médio + Assimetria ALTA + Cor VARIÁVEL',  'Risco ALTO',      'Variação estrutural e cromática'),
    ('R9','P médio + Borda IRREG + Estrutura COMPL',   'Risco ALTO',      'Dois critérios morfológicos'),
    ('R10','P médio + Assimetria ALTA',                'Risco MODERADO',  'Rede incerta, CAM assimétrico'),
    ('R11','P médio + Borda IRREGULAR',                'Risco MODERADO',  'Rede incerta, borda suspeita'),
    ('R12','P médio (sozinho)',                        'Risco MODERADO',  'Rede indecisa — avaliar clinicamente'),
    ('R13','P médio + Assimetria BAIXA + Borda REG',   'Risco BAIXO',     'Rede incerta mas morfologia benigna'),
    ('R14','P médio + Assimetria BAIXA + Cor UNIF',    'Risco BAIXO',     'Rede incerta mas CAM uniforme'),
    ('R15','Assimetria ALTA + Borda IRREG + Cor VAR',  'Risco MUITO ALTO','Tríade ABCD clássica do melanoma'),
    ('R16','Cor VARIÁVEL + Estrutura COMPLEXA',        'Risco MODERADO',  'Heterogeneidade interna'),
]
cores_r={'MUITO ALTO':('FFC7CE','9C0006'),'ALTO':('FFD7B5','7B3800'),
         'MODERADO':('FFEB9C','9C5700'),'BAIXO':('E2EFDA','375623'),'MUITO BAIXO':('C6EFCE','276221')}
for ri5,(rid,cond,conc,logica) in enumerate(regras_doc,3):
    for ci,val in enumerate([rid,cond,conc,logica],1):
        c=ws3.cell(row=ri5,column=ci,value=val); c.border=bdr(); c.font=Font(size=9)
        c.alignment=aln(h='center' if ci==1 else 'left')
        if ri5%2==1: c.fill=fill('F5F5F5')
        if ci==3:
            nivel_k=[k for k in cores_r if k in conc]
            if nivel_k:
                bg,fg=cores_r[nivel_k[0]]
                c.fill=fill(bg); c.font=Font(bold=True,color=fg,size=9)
    ws3.row_dimensions[ri5].height=36

OUTPUT=f'/kaggle/working/relatorio_fuzzy_v3_{MAX_IMAGES}imgs.xlsx'
wb.save(OUTPUT)
print(f'XLSX: {OUTPUT}')
print('Abas: Resultados | Métricas | Regras Fuzzy v3')

## 12. Relatório XLSX para uso clínico

Gera um arquivo `.xlsx` com três abas, pensado para ser consumido por equipes
clínicas sem necessidade de programação:

- **Resultados** — uma linha por lesão de teste: todos os scores, regras
  ativadas e explicação em linguagem natural, com formatação condicional por
  nível de risco.
- **Métricas** — comparação resumida entre baseline, rede pura e fuzzy.
- **Regras Fuzzy v3** — as 16 regras com o racional clínico de cada uma.


In [ ]:
# ══════════════════════════════════════════════════════════
# Cole essa célula no seu notebook Kaggle
# Gera figura de Grad-CAM para o artigo — 2 linhas x 4 colunas
# Linha 1: 2 malignos + 2 benignos (imagem original)
# Linha 2: os mesmos com Grad-CAM sobreposto + scores ABCD
# Salva em /kaggle/working/gradcam_paper.pdf e .png
# ══════════════════════════════════════════════════════════
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# Seleciona 2 malignos e 2 benignos do conjunto de teste
# Critério: pega casos com score fuzzy alto (malignos) e baixo (benignos)
# para mostrar o contraste visual

df_mal = df_res[df_res['is_malignant'] == 1].sort_values('fuzzy_score', ascending=False)
df_ben = df_res[df_res['is_malignant'] == 0].sort_values('fuzzy_score', ascending=True)

# Pega 2 de cada — com scores bem definidos (não casos limítrofes)
casos_mal = df_mal.head(2).reset_index(drop=True)
casos_ben = df_ben.head(2).reset_index(drop=True)
casos = pd.concat([casos_mal, casos_ben]).reset_index(drop=True)

DX_FULL = {
    'mel':'Melanoma','nv':'Nevo Melanocítico',
    'bcc':'Carcinoma Basocelular','akiec':'Ceratose Actínica',
    'bkl':'Ceratose Seborreica','df':'Dermatofibroma','vasc':'Lesão Vascular'
}

fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor('white')

gs = GridSpec(3, 4, figure=fig,
              hspace=0.05, wspace=0.08,
              top=0.88, bottom=0.18)

# Título do artigo
fig.suptitle(
    'Grad-CAM Saliency Maps and Grad-CAM-Derived ABCD Scores\n'
    'Columns 1–2: Malignant lesions | Columns 3–4: Benign lesions',
    fontsize=13, fontweight='bold'
)
for col_idx, (_, row) in enumerate(casos.iterrows()):
    path = df_test[df_test['image_id'] == row['image_id']]['image_path'].values[0]

    # Extrai Grad-CAM e features
    p_mal, cam, img_orig, mask, A, B, C, D = extrator.extrair(path)
    s, Bn, Ai, Pi = inferir_v3(A, B, C, D, p_mal)  # [COR10]
    score = s

    # Gera heatmap colorido
    hm      = cv2.applyColorMap((cam*255).astype(np.uint8), cv2.COLORMAP_JET)
    hm_rgb  = cv2.cvtColor(hm, cv2.COLOR_BGR2RGB)
    overlay = (img_orig * 0.45 + hm_rgb * 0.55).astype(np.uint8)

    is_mal  = bool(row['is_malignant'])
    cor_brd = '#C0392B' if is_mal else '#27AE60'
    label   = 'MALIGNANT' if is_mal else 'BENIGN'
    dx_nome = DX_FULL.get(row['dx'], row['dx'])

    # ── Linha 0: imagem original ──────────────────────────────
    ax0 = fig.add_subplot(gs[0, col_idx])
    ax0.imshow(img_orig)
    ax0.axis('off')
    # Borda colorida por classe
    for spine in ax0.spines.values():
        spine.set_edgecolor(cor_brd)
        spine.set_linewidth(3)
    ax0.set_title(
        f'{dx_nome}\n{label}',
        fontsize=9, fontweight='bold', color=cor_brd, pad=4
    )

    # ── Linha 1: Grad-CAM sobreposto ──────────────────────────
    ax1 = fig.add_subplot(gs[1, col_idx])
    ax1.imshow(overlay)
    ax1.axis('off')
    for spine in ax1.spines.values():
        spine.set_edgecolor(cor_brd)
        spine.set_linewidth(3)

    # ── Linha 2: scores ABCD em barras horizontais ────────────
    ax2 = fig.add_subplot(gs[2, col_idx])
    
    feat_names = ['P(rede)', 'A (real)', 'B (norm)', 'C', 'D', 'Risco/10']
    feat_vals  = [p_mal, Ai, Bn, C, D, score/10.0]
    bar_colors = ['#8E44AD','#3498DB','#27AE60','#E67E22','#E74C3C','#1B3A5C']

    bars = ax2.barh(feat_names, feat_vals, color=bar_colors,
                    edgecolor='white', height=0.6)
    ax2.set_xlim(0, 1.05)
    ax2.axvline(0.5, color='gray', lw=0.8, linestyle='--', alpha=0.5)
    ax2.set_xlabel('Score', fontsize=8)
    ax2.tick_params(axis='y', labelsize=8)
    ax2.tick_params(axis='x', labelsize=7)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    ax2.set_facecolor('#FAFAFA')

    # Valores nas barras
    for bar, val in zip(bars, feat_vals):
        ax2.text(min(val + 0.02, 0.98), bar.get_y() + bar.get_height()/2,
                 f'{val:.2f}', va='center', fontsize=7, fontweight='bold')

    # Nível de risco como título do painel
    nivel = nivel_risco(score)
    cor_nivel = {'MUITO BAIXO':'#27AE60','BAIXO':'#82E0AA',
                 'MODERADO':'#F39C12','ALTO':'#E74C3C','MUITO ALTO':'#922B21'}
    ax2.set_title(f'Risk: {nivel}', fontsize=8, fontweight='bold',
                  color=cor_nivel.get(nivel,'#333333'), pad=3)


# Labels das linhas
for row_idx, row_label in enumerate(['Original image', 'Grad-CAM overlay', 'ABCD scores']):
    ax_ref = fig.add_subplot(gs[row_idx, 0])
    ax_ref.text(-0.22, 0.5, row_label,
                transform=ax_ref.transAxes,
                fontsize=9, fontweight='bold', rotation=90,
                va='center', ha='center', color='#444444')

# Colorbar do Grad-CAM
cax = fig.add_axes([0.15, 0.10, 0.70, 0.018])
sm  = plt.cm.ScalarMappable(cmap='jet', norm=plt.Normalize(0, 1))
sm.set_array([])
cbar = fig.colorbar(sm, cax=cax, orientation='horizontal')
cbar.set_label('Grad-CAM saliency (0 = low attention → 1 = high attention)',
               fontsize=9)
cbar.set_ticks([0, 0.5, 1])
cbar.set_ticklabels(['Low', 'Medium', 'High'])

plt.savefig('/kaggle/working/gradcam_paper.pdf',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/kaggle/working/gradcam_paper.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Salvo: /kaggle/working/gradcam_paper.pdf')
print('Salvo: /kaggle/working/gradcam_paper.png')

## 13. Figura Grad-CAM para o artigo

Seleciona 2 casos malignos (maior score fuzzy) e 2 benignos (menor score
fuzzy) do conjunto de teste, montando uma figura 3 linhas × 4 colunas: imagem
original, overlay Grad-CAM, e barras horizontais com os scores ABCD + risco.
Salva em `gradcam_paper.pdf`/`.png` — corresponde à Figura 2 do artigo.


In [ ]:
# ══════════════════════════════════════════════════════════
# VISUALIZAÇÃO DAS FUNÇÕES DE PERTINÊNCIA — FUZZY v3
# Executa APÓS build_fuzzy_v3 e calibração (params_cal)
# ══════════════════════════════════════════════════════════

def plot_mfs_v3(params):
    u = np.arange(0, 1.001, 0.002)
    r = np.arange(0, 10.1,  0.1)

    def get_mf_points(p):
        vmin, vmax, bt, mc, ab = p
        span    = vmax - vmin
        overlap = max(span * 0.25, 1e-4)
        bt_low  = float(np.clip(bt - overlap*0.5, vmin,        mc - 1e-4))
        bt_high = float(np.clip(bt + overlap*0.5, bt_low+1e-4, mc))
        ab_low  = float(np.clip(ab - overlap*0.5, mc,          vmax-1e-4))
        ab_high = float(np.clip(ab + overlap*0.5, ab_low+1e-4, vmax))
        baixo = fuzz.trapmf(u, [vmin, vmin, bt_low,  bt_high])
        medio = fuzz.trimf (u, [bt_low, mc, ab_high])
        alto  = fuzz.trapmf(u, [ab_low, ab_high, vmax, vmax])
        return baixo, medio, alto, bt_low, bt_high, ab_low, ab_high, mc

    mb  = fuzz.trapmf(r, [0,   0,   1.5, 3.0])
    bx  = fuzz.trimf (r, [1.5, 3.0, 4.5])
    md  = fuzz.trimf (r, [3.0, 5.0, 7.0])
    al  = fuzz.trimf (r, [5.5, 7.0, 8.5])
    ma  = fuzz.trapmf(r, [7.0, 8.5, 10,  10])

    CORES  = ['#2ECC71', '#3498DB', '#E67E22']
    FEAT_NAMES = [
        ('A', 'Asymmetry (A)',  params['A']),
        ('B', 'Border (B)',     params['B']),
        ('C', 'Color (C)',      params['C']),
        ('D', 'Structure (D)',  params['D']),
        ('P', 'Network score (P)', params['P']),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle('Funções de Pertinência', fontsize=15, fontweight='bold', y=0.99)

    axs = [axes[0,0], axes[0,1], axes[0,2], axes[1,0], axes[1,1]]
    for ax, (k, nome, p) in zip(axs, FEAT_NAMES):
        baixo, medio, alto, bt_low, bt_high, ab_low, ab_high, mc = get_mf_points(p)
        vmin, vmax, bt, _, ab = p

        ax.fill_between(u, baixo, alpha=0.12, color=CORES[0])
        ax.fill_between(u, medio, alpha=0.12, color=CORES[1])
        ax.fill_between(u, alto,  alpha=0.12, color=CORES[2])
        ax.plot(u, baixo, color=CORES[0], lw=2, label='baixo')
        ax.plot(u, medio, color=CORES[1], lw=2, label='médio')
        ax.plot(u, alto,  color=CORES[2], lw=2, label='alto')

        # Apenas linhas de transição, sem rótulos numéricos
        for xv in [bt_low, bt_high, mc, ab_low, ab_high]:
            ax.axvline(xv, color='gray', lw=0.7, ls='--', alpha=0.4)

        ax.set_title(nome, fontsize=11, fontweight='bold')
        ax.set_xlim(vmin - 0.02, vmax + 0.02)
        ax.set_ylim(-0.05, 1.1)
        ax.set_yticks([0, 0.5, 1])
        ax.tick_params(labelsize=8)
        ax.grid(alpha=0.15)

    axs[0].legend(fontsize=8, loc='upper right', frameon=False)

    ax = axes[1, 2]
    CORES_OUT = ['#1ABC9C','#2ECC71','#F39C12','#E74C3C','#8E44AD']
    LABELS_OUT = ['muito baixo','baixo','médio','alto','muito alto']
    MFS_OUT = [mb, bx, md, al, ma]
    for mf_out, label, cor in zip(MFS_OUT, LABELS_OUT, CORES_OUT):
        ax.fill_between(r, mf_out, alpha=0.12, color=cor)
        ax.plot(r, mf_out, lw=2, label=label, color=cor)
    for xv in [1.5, 3.0, 4.5, 5.5, 7.0, 8.5]:
        ax.axvline(xv, color='gray', lw=0.7, ls='--', alpha=0.4)
    ax.set_title('Risco de saída', fontsize=11, fontweight='bold')
    ax.set_xlim(-0.2, 10.2)
    ax.set_ylim(-0.05, 1.1)
    ax.set_yticks([0, 0.5, 1])
    ax.tick_params(labelsize=8)
    ax.legend(fontsize=7, loc='upper right', frameon=False)
    ax.grid(alpha=0.15)

    plt.tight_layout()
    plt.savefig('/kaggle/working/mfs_fuzzy_v3.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Salvo: /kaggle/working/mfs_fuzzy_v3.png')


plot_mfs_v3(params_cal)